# 2. Nettoyage des données

Cette étape consiste à améliorer la qualité et la cohérence du jeu de données avant son utilisation pour la modélisation. Nous examinerons les particularités identifiées lors de l'analyse exploratoire, notamment les modalités `unknown` et les catégories peu représentées. Des traitements adaptés seront appliqués afin d'obtenir des données plus cohérentes, tout en préservant les informations utiles à la prédiction de la souscription bancaire.

In [2]:
import pandas as pd
import numpy as np

# Chargement des données brutes
df = pd.read_csv("../data/raw/bank-additional.csv", sep=";")

# Copie destinée au nettoyage
df_clean = df.copy()

print("Dimensions initiales :", df_clean.shape)

Dimensions initiales : (4119, 21)


### 2.1. Vérification des modalités catégorielles

Avant d'effectuer le nettoyage, nous examinons la répartition des modalités des variables catégorielles concernées. Cette vérification permet d'identifier les catégories à traiter et de contrôler les effets des transformations appliquées par la suite.

In [4]:
# Répartition des modalités avant nettoyage
variables_a_examiner = [
    "housing", "loan", "marital", "default",
    "education", "job"
]

for variable in variables_a_examiner:
    print(f"\n--- {variable} ---")
    print(df_clean[variable].value_counts(dropna=False))


--- housing ---
housing
yes        2175
no         1839
unknown     105
Name: count, dtype: int64

--- loan ---
loan
no         3349
yes         665
unknown     105
Name: count, dtype: int64

--- marital ---
marital
married     2509
single      1153
divorced     446
unknown       11
Name: count, dtype: int64

--- default ---
default
no         3315
unknown     803
yes           1
Name: count, dtype: int64

--- education ---
education
university.degree      1264
high.school             921
basic.9y                574
professional.course     535
basic.4y                429
basic.6y                228
unknown                 167
illiterate                1
Name: count, dtype: int64

--- job ---
job
admin.           1012
blue-collar       884
technician        691
services          393
management        324
retired           166
self-employed     159
entrepreneur      148
unemployed        111
housemaid         110
student            82
unknown            39
Name: count, dtype: int64


### 2.2. Traitement des modalités inconnues

Les variables `housing` et `loan` comportent des observations dont la modalité est `unknown`. Afin d'harmoniser ces variables, ces observations seront regroupées dans la catégorie `no`, conformément aux règles de traitement retenues. Les effectifs seront vérifiés après transformation pour s'assurer que le traitement a été correctement appliqué.

In [5]:
# Effectifs avant traitement
print("Housing avant traitement :")
print(df_clean["housing"].value_counts())

print("\nLoan avant traitement :")
print(df_clean["loan"].value_counts())

# Remplacement des valeurs unknown par no
df_clean["housing"] = df_clean["housing"].replace("unknown", "no")
df_clean["loan"] = df_clean["loan"].replace("unknown", "no")

# Vérification après traitement
print("\nHousing après traitement :")
print(df_clean["housing"].value_counts())

print("\nLoan après traitement :")
print(df_clean["loan"].value_counts())

Housing avant traitement :
housing
yes        2175
no         1839
unknown     105
Name: count, dtype: int64

Loan avant traitement :
loan
no         3349
yes         665
unknown     105
Name: count, dtype: int64

Housing après traitement :
housing
yes    2175
no     1944
Name: count, dtype: int64

Loan après traitement :
loan
no     3454
yes     665
Name: count, dtype: int64


### 2.3. Traitement des variables `marital` et `default`

Le traitement des modalités inconnues dépend de la variable concernée. Pour `marital`, les observations `unknown` seront regroupées dans la catégorie `single`. Pour `default`, la modalité `yes` sera regroupée dans `unknown`, qui sera conservée comme catégorie distincte de `no`.

Ces transformations seront suivies d'une vérification des effectifs afin de contrôler leur bonne application.

In [6]:
# Effectifs avant traitement
print("Marital avant traitement :")
print(df_clean["marital"].value_counts())

print("\nDefault avant traitement :")
print(df_clean["default"].value_counts())

# Traitement des modalités
df_clean["marital"] = df_clean["marital"].replace("unknown", "single")
df_clean["default"] = df_clean["default"].replace("yes", "unknown")

# Vérification après traitement
print("\nMarital après traitement :")
print(df_clean["marital"].value_counts())

print("\nDefault après traitement :")
print(df_clean["default"].value_counts())

Marital avant traitement :
marital
married     2509
single      1153
divorced     446
unknown       11
Name: count, dtype: int64

Default avant traitement :
default
no         3315
unknown     803
yes           1
Name: count, dtype: int64

Marital après traitement :
marital
married     2509
single      1164
divorced     446
Name: count, dtype: int64

Default après traitement :
default
no         3315
unknown     804
Name: count, dtype: int64


### 2.4. Vérification après nettoyage

Après le traitement des modalités inconnues, nous vérifions que les transformations ont été correctement appliquées et que le jeu de données ne présente pas d'autres problèmes évidents de qualité. Cette vérification porte notamment sur les valeurs manquantes, les doublons et les modalités restantes dans les variables traitées.

In [7]:
# Vérification des valeurs manquantes
print("Valeurs manquantes par variable :")
print(df_clean.isna().sum().sum())

# Vérification des doublons
print("\nNombre de lignes dupliquées :")
print(df_clean.duplicated().sum())

# Vérification des modalités après traitement
variables_traitees = ["housing", "loan", "marital", "default"]

for variable in variables_traitees:
    print(f"\n--- {variable} ---")
    print(df_clean[variable].value_counts(dropna=False))

# Vérification des dimensions
print("\nDimensions après traitement :", df_clean.shape)

Valeurs manquantes par variable :
0

Nombre de lignes dupliquées :
0

--- housing ---
housing
yes    2175
no     1944
Name: count, dtype: int64

--- loan ---
loan
no     3454
yes     665
Name: count, dtype: int64

--- marital ---
marital
married     2509
single      1164
divorced     446
Name: count, dtype: int64

--- default ---
default
no         3315
unknown     804
Name: count, dtype: int64

Dimensions après traitement : (4119, 21)


### 2.5. Bilan des transformations

Cette section récapitule les modifications apportées aux variables catégorielles et vérifie que le nettoyage a conservé la structure du jeu de données. L'objectif est de s'assurer que les traitements sont conformes aux règles définies et que les données restent exploitables pour les étapes suivantes.

In [8]:
# Comparaison des dimensions avant et après nettoyage
print("Dimensions initiales :", df.shape)
print("Dimensions après nettoyage :", df_clean.shape)

# Comparaison des modalités pour les variables traitées
for variable in ["housing", "loan", "marital", "default"]:
    print(f"\n--- {variable} ---")
    print("Avant :")
    print(df[variable].value_counts())
    print("Après :")
    print(df_clean[variable].value_counts())

Dimensions initiales : (4119, 21)
Dimensions après nettoyage : (4119, 21)

--- housing ---
Avant :
housing
yes        2175
no         1839
unknown     105
Name: count, dtype: int64
Après :
housing
yes    2175
no     1944
Name: count, dtype: int64

--- loan ---
Avant :
loan
no         3349
yes         665
unknown     105
Name: count, dtype: int64
Après :
loan
no     3454
yes     665
Name: count, dtype: int64

--- marital ---
Avant :
marital
married     2509
single      1153
divorced     446
unknown       11
Name: count, dtype: int64
Après :
marital
married     2509
single      1164
divorced     446
Name: count, dtype: int64

--- default ---
Avant :
default
no         3315
unknown     803
yes           1
Name: count, dtype: int64
Après :
default
no         3315
unknown     804
Name: count, dtype: int64


### 2.6. Enregistrement des données nettoyées

Les premières corrections sont terminées. Nous enregistrons le jeu de données dans `data/interim/` pour conserver une version nettoyée et pouvoir la reprendre pour la suite du projet. Le fichier brut reste inchangé.

In [9]:
from pathlib import Path

# Création du dossier de destination si nécessaire
dossier_sortie = Path("../data/interim")
dossier_sortie.mkdir(parents=True, exist_ok=True)

# Enregistrement des données nettoyées
df_clean.to_csv(
    dossier_sortie / "bank_marketing_clean.csv",
    index=False,
    sep=";"
)

print("Données nettoyées enregistrées.")
print("Dimensions :", df_clean.shape)

Données nettoyées enregistrées.
Dimensions : (4119, 21)
